# CineIQ — Content-Based Filtering

Content-based filtering recommends items similar to what a user has liked before, based on item features.

## Approach
- **TF-IDF Vectorization**: Convert genre labels into numerical feature vectors.
- **Cosine Similarity**: Measure pairwise similarity between movies based on their genre vectors.
- **Recommendation**: For a given movie, return the most similar movies (excluding itself).

In [1]:
# Imports
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

# Load
movies = pd.read_csv('../data/processed/movies.csv')
print(movies.head())

# Prepare genre features
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)
print(movies[['title', 'genres_clean']].head(10))

   movieId                               title                        genres
0        1                    Toy Story (1995)   Animation|Children's|Comedy
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy
2        3             Grumpier Old Men (1995)                Comedy|Romance
3        4            Waiting to Exhale (1995)                  Comedy|Drama
4        5  Father of the Bride Part II (1995)                        Comedy
                                title                  genres_clean
0                    Toy Story (1995)   Animation Children's Comedy
1                      Jumanji (1995)  Adventure Children's Fantasy
2             Grumpier Old Men (1995)                Comedy Romance
3            Waiting to Exhale (1995)                  Comedy Drama
4  Father of the Bride Part II (1995)                        Comedy
5                         Heat (1995)         Action Crime Thriller
6                      Sabrina (1995)                Comedy Ro

## Cleaning Genre Features
The MovieLens dataset uses pipe-delimited genres (e.g., `Animation|Children's|Comedy`). We replace `|` with spaces for TF-IDF tokenization.

In [2]:
# TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])
print("TF-IDF matrix shape:", tfidf_matrix.shape)

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print("Cosine sim matrix shape:", cosine_sim.shape)

# Index map
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

TF-IDF matrix shape: (3883, 20)
Cosine sim matrix shape: (3883, 3883)


## Building the Similarity Matrix
We fit a TF-IDF vectorizer (ignoring English stop words) on the cleaned genre text and compute the full cosine similarity matrix. The result is a 3883×3883 matrix where each entry represents genre similarity between two movies.

In [3]:
# Recommendation function
def get_content_recommendations(title, n=10):
    if title not in indices:
        return f"Movie '{title}' not found"
    
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]  # skip itself
    movie_indices = [i[0] for i in sim_scores]
    
    return movies[['title', 'genres']].iloc[movie_indices]

## Recommendation Function
`get_content_recommendations(title, n=10)` looks up the input movie, retrieves its similarity scores, and returns the top-n most similar movies (skipping the input itself).

In [4]:
# Test it
print(get_content_recommendations("Toy Story (1995)"))

# Save artifacts
pickle.dump(cosine_sim, open('../models/cosine_sim.pkl', 'wb'))
pickle.dump(indices, open('../models/indices.pkl', 'wb'))
movies.to_csv('../data/processed/movies.csv', index=False)
print("Saved.")

                                               title  \
1050          Aladdin and the King of Thieves (1996)   
2072                        American Tail, An (1986)   
2073      American Tail: Fievel Goes West, An (1991)   
2285                       Rugrats Movie, The (1998)   
2286                            Bug's Life, A (1998)   
3045                              Toy Story 2 (1999)   
3542                           Saludos Amigos (1943)   
3682                              Chicken Run (2000)   
3685  Adventures of Rocky and Bullwinkle, The (2000)   
12                                      Balto (1995)   

                           genres  
1050  Animation|Children's|Comedy  
2072  Animation|Children's|Comedy  
2073  Animation|Children's|Comedy  
2285  Animation|Children's|Comedy  
2286  Animation|Children's|Comedy  
3045  Animation|Children's|Comedy  
3542  Animation|Children's|Comedy  
3682  Animation|Children's|Comedy  
3685  Animation|Children's|Comedy  
12           Animation|

## Testing & Saving
Testing with "Toy Story (1995)" returns animated/children's movies as expected — all genre-related. We save the cosine similarity matrix and index mapping for use by the ensemble model.